# nstCollExamples

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `full`
- Workflow family: `signal`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/nstCollExamples.md`


Notebook source link: [nstCollExamples.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/nstCollExamples.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "nstCollExamples"
FAMILY = "signal"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"nstCollExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"nstCollExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"nstCollExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"nstCollExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# nstCollExamples: collection masking and single-neuron extraction.
from nstat.compat.matlab import History, nspikeTrain, nstColl

trains = []
for i in range(20):
    spk = np.sort(rng.random(100))
    unit = nspikeTrain(spike_times=spk, t_start=0.0, t_end=1.0, name=f"Neuron{i+1}")
    unit.setName(f"Neuron{i+1}")
    trains.append(unit)
spikeColl = nstColl(trains)

fig1 = plt.figure(figsize=(9.0, 4.0))
spikeColl.plot()
plt.title(f"{TOPIC}: full collection raster")
plt.xlabel("time [s]")
plt.tight_layout()
plt.show()

spikeColl.setMask([1, 4, 7])
fig2 = plt.figure(figsize=(9.0, 3.6))
spikeColl.plot()
plt.title("Masked collection raster (units 1, 4, 7)")
plt.xlabel("time [s]")
plt.tight_layout()
plt.show()

n1 = spikeColl.getNST(0)
sig_1ms = n1.getSigRep(binSize_s=0.001, mode="binary")
sig_10ms = n1.getSigRep(binSize_s=0.01, mode="binary")

fig3, axes = plt.subplots(3, 1, figsize=(9.0, 6.0), sharex=False)
plt.sca(axes[0])
n1.plot()
axes[0].set_title("Unit 1 spikes")
axes[0].set_xlabel("time [s]")
axes[1].step(np.arange(sig_1ms.size) * 0.001, sig_1ms, where="post", color="tab:blue")
axes[1].set_title("Unit 1 binary 1 ms")
axes[2].step(np.arange(sig_10ms.size) * 0.01, sig_10ms, where="post", color="tab:green")
axes[2].set_title("Unit 1 binary 10 ms")
axes[2].set_xlabel("time [s]")
plt.tight_layout()
plt.show()

masked = spikeColl.getIndFromMask()
history = History(bin_edges_s=np.array([0.0, 0.01, 0.03], dtype=float))
spikes = n1
H = history.computeHistory(spikes.spike_times, np.arange(0.0, 1.0, 0.01))
assert H.ndim == 2 and H.shape[1] == history.n_bins
assert spikes.spike_times.size > 5
assert len(masked) == 3
assert spikeColl.getNumUnits() == 20

CHECKPOINT_METRICS = {
    "num_units": float(spikeColl.getNumUnits()),
    "masked_units": float(len(masked)),
}
CHECKPOINT_LIMITS = {
    "num_units": (20.0, 20.0),
    "masked_units": (3.0, 3.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
